# 20 Routine Quick Job Cleanup Template

Read-only planning template for routine cleanup and repair jobs.


In [1]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [2]:
# Notebook parameters (edit here for local runs).
ADAMACS_MAX_JOB_ROWS = 50
ADAMACS_INITIALS = "SM"
ADAMACS_DATE_FROM = "2025-01-01"
ADAMACS_DATE_TO = None  # Optional inclusive upper bound, e.g. "2025-12-31"

print("ADAMACS_MAX_JOB_ROWS =", ADAMACS_MAX_JOB_ROWS)
print("ADAMACS_INITIALS     =", ADAMACS_INITIALS)
print("ADAMACS_DATE_FROM    =", ADAMACS_DATE_FROM)
print("ADAMACS_DATE_TO      =", ADAMACS_DATE_TO)


ADAMACS_MAX_JOB_ROWS = 50
ADAMACS_INITIALS     = SM
ADAMACS_DATE_FROM    = 2025-01-01
ADAMACS_DATE_TO      = None


In [3]:
import pandas as pd

from adamacs.pipeline import (
    subject,
    session,
    scan,
    event,
    trial,
    imaging,
    behavior,
    model,
    denoising,
    mocap,
    disk
)

ALLOW_DB_WRITES = False
MAX_ROWS = int(ADAMACS_MAX_JOB_ROWS)
INITIALS = ADAMACS_INITIALS
DATE_FROM = ADAMACS_DATE_FROM
DATE_TO = ADAMACS_DATE_TO

print("ALLOW_DB_WRITES:", ALLOW_DB_WRITES)
print("INITIALS:", INITIALS)
print("DATE_FROM:", DATE_FROM)
print("DATE_TO:", DATE_TO)


ALLOW_DB_WRITES: False
INITIALS: SM
DATE_FROM: 2025-01-01
DATE_TO: None



## 1) Job overview by schema

Fast check of `status` counts in the key operational schemas.


In [4]:

schema_map = {
    "imaging": imaging.schema,
    "model": model.schema,
    "denoising": denoising.schema,
    "mocap": mocap.schema,
}

status_frames = []
for name, schema in schema_map.items():
    try:
        jobs = schema.jobs
        frame = dj.U("status").aggr(jobs, n="count(*)").fetch(format="frame").reset_index()
        frame.insert(0, "schema", name)
        status_frames.append(frame)
    except Exception as exc:
        status_frames.append(pd.DataFrame([{"schema": name, "status": "error", "n": f"unavailable: {exc}"}]))

pd.concat(status_frames, ignore_index=True)


,schema,status,n
0,imaging,reserved,1
1,imaging,error,101
2,model,reserved,2
3,model,error,17
4,mocap,error,122



### 1b) Direct jobs tables (DataJoint-native view)

Raw `schema.jobs` relations without `fetch`/pandas conversion.


In [5]:

job_relations = {
    "imaging.jobs": imaging.schema.jobs,
    "model.jobs": model.schema.jobs,
    "denoising.jobs": denoising.schema.jobs,
    "mocap.jobs": mocap.schema.jobs,
}

for label, relation in job_relations.items():
    print(f"\n### {label}")
    try:
        display(relation)
    except Exception as exc:
        print(f"{label}: unavailable ({exc})")



### imaging.jobs


table_name className of the table,key_hash key hash,"status if tuple is missing, the job is available",key structure containing the key,error_message error message returned if failed,error_stack error stack if failed,user database user,host system hostname,pid system process id,connection_id connection_id(),timestamp automatic timestamp
_motion_correction,15ddd0a701fb0b1be3fbf84cd181ce77,error,=BLOB=,KeyError: 'Vcorr',=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-13 11:42:29
_motion_correction,707c9e6cfd169a06161fd0ef76736c0d,error,=BLOB=,KeyError: 'Vcorr',=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-13 11:41:48
_motion_correction,ac8102453b0b93b5f610c096b8d835f5,error,=BLOB=,KeyError: 'Vcorr',=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-13 11:41:48
_motion_correction,bd0a58d00a72bc5f513ff938fefc4093,error,=BLOB=,KeyError: 'Vcorr',=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-13 11:41:48
_motion_correction,c50c23fc58df63a1bcc43bf7af8af234,error,=BLOB=,KeyError: 'Vcorr',=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-13 11:41:48
__activity,070df51727a9d3916d266c772a54a94d,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FXY0HRX-scan9FXY0HRX-10-2-suite2p_deconvolution' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-20 10:52:26
__activity,7a689f2f3d02bb1f16c890f79617755c,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FXY0HRX-scan9FXY0HRX-10-3-suite2p_deconvolution' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-20 11:01:58
__fluorescence,06165011d4c0e6e25fd56902da29b551,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FXY0HRX-scan9FXY0HRX-10-2' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-20 10:52:24
__fluorescence,6d6d9337d109362903c260162a2f2103,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FY3GJQ3-scan9FY3GJQ3-10-2' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-22 15:23:27
__fluorescence,a2a77221c1127b777d62ff373714e3e1,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FXY0HRX-scan9FXY0HRX-10-3' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,3571176,96922,2026-01-20 11:01:56



### model.jobs


table_name className of the table,key_hash key hash,"status if tuple is missing, the job is available",key structure containing the key,error_message error message returned if failed,error_stack error stack if failed,user database user,host system hostname,pid system process id,connection_id connection_id(),timestamp automatic timestamp
_recording_info_new,342c0b7cd28e2830fcd7e5212a3373b6,reserved,=BLOB=,,=BLOB=,jisooj@172.25.70.3,ibehavegpu1,978489,201229,2026-01-22 14:14:23
_recording_info_new,af7b4fff4c72c7b43457d9e995fd657c,error,=BLOB=,ZeroDivisionError: division by zero,=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-01-25 02:43:49
_recording_info_new,f0aba20c0e0c691bb55f3a24d3f953db,reserved,=BLOB=,,=BLOB=,jisooj@172.25.70.3,ibehavegpu1,978489,201229,2026-01-22 14:14:36
__pose_estimation_new,03000837905ffaa8c74a7b9163311cdf,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-03 22:09:46
__pose_estimation_new,0adcb3f62a9c1cb5ff81e52613bc2300,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-03 21:10:39
__pose_estimation_new,12d99ea90969fe1823edabd8f0ea27bb,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-03 21:10:39
__pose_estimation_new,302a96f1bfd13a5df9c9470e052edc47,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-01-30 09:56:16
__pose_estimation_new,535a4e8ee18ce41237f6d86d2e193404,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-04 13:49:05
__pose_estimation_new,5ab9552e36f5329f9577193bb7756675,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-03 21:10:40
__pose_estimation_new,a5b870f1361e2f702eb4254e08bf3173,error,=BLOB=,ValueError: invalid literal for int() with base 10: 'Failed to initialize NVML: Driver/library version mismatch',=BLOB=,tobiasr@172.25.70.3,ibehavegpu1,5841,146257,2026-02-01 17:46:39



### denoising.jobs


table_name className of the table,key_hash key hash,"status if tuple is missing, the job is available",key structure containing the key,error_message error message returned if failed,error_stack error stack if failed,user database user,host system hostname,pid system process id,connection_id connection_id(),timestamp automatic timestamp



### mocap.jobs


table_name className of the table,key_hash key hash,"status if tuple is missing, the job is available",key structure containing the key,error_message error message returned if failed,error_stack error stack if failed,user database user,host system hostname,pid system process id,connection_id connection_id(),timestamp automatic timestamp
_mocap_recording_info,081d0c0d2fc62af6fd82133cf45a5cae,error,=BLOB=,StopIteration,=BLOB=,tobiasr@172.25.64.3,tatchu3,2689115,13359,2025-12-05 17:08:41
_mocap_recording_info,11c068290a20be4a75b276ecee809027,error,=BLOB=,StopIteration,=BLOB=,tobiasr@172.25.64.3,tatchu3,2689115,13359,2025-12-05 17:08:46
_mocap_recording_info,15724c3cfd5583e65e42d1dcfe7c6821,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FUCT4QE-scan9FUCT4QE' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,5013,7,2025-08-13 18:20:04
_mocap_recording_info,15a9b67fb950fbff51c6d44632b1bd77,error,=BLOB=,EmptyDataError: No columns to parse from file,=BLOB=,tobiasr@172.25.64.3,tatchu3,676252,542014,2025-08-06 14:46:51
_mocap_recording_info,164855b1b86847a066ba21fbdc15bd35,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FU1BEUM-scan9FU1BEUM' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,5013,7,2025-08-18 17:36:28
_mocap_recording_info,1834cb0760eb6bf048aab64f37653545,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FU4ATBA-scan9FU4ATBA' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,5013,7,2025-08-19 11:58:03
_mocap_recording_info,27ce800099039d8a5005b009c064189b,error,=BLOB=,StopIteration,=BLOB=,tobiasr@172.25.64.3,tatchu3,2689115,13359,2025-12-05 17:08:44
_mocap_recording_info,29540265f03ccc3ab7eb8c0d294c2974,error,=BLOB=,StopIteration,=BLOB=,tobiasr@172.25.64.3,tatchu3,2689115,13359,2025-12-05 17:07:41
_mocap_recording_info,2961eacf251c2327872ac9ba4b692b1e,error,=BLOB=,StopIteration,=BLOB=,tobiasr@172.25.64.3,tatchu3,2689115,13359,2025-12-05 17:07:33
_mocap_recording_info,2b4342d5c9c5b4831f52e82e6bd20ac0,error,=BLOB=,"DuplicateError: (""Duplicate entry 'sess9FUAAX2M-scan9FUAAX2M' for key 'PRIMARY'"", 'To ignore duplicate entries in insert, set skip_duplicates=True')",=BLOB=,tobiasr@172.25.64.3,tatchu3,5013,7,2025-08-19 09:31:32



## 2) Recent error jobs (read-only triage)


In [6]:

def recent_jobs(schema, status="error", limit=50):
    rows = []
    try:
        query = schema.jobs & f'status="{status}"'
        fetched = query.fetch(
            "timestamp",
            "status",
            "host",
            "key_hash",
            "error_message",
            "error_stack",
            "key",
            as_dict=True,
            order_by="timestamp desc",
            limit=limit,
        )
        for row in fetched:
            rows.append(
                {
                    "timestamp": row.get("timestamp"),
                    "status": row.get("status"),
                    "host": row.get("host"),
                    "key_hash": row.get("key_hash"),
                    "error_message": str(row.get("error_message", ""))[:180],
                    "key_preview": str(row.get("key", {}))[:180],
                    "stack_preview": str(row.get("error_stack", ""))[:220],
                }
            )
    except Exception as exc:
        rows.append({"error": f"Could not fetch {status} jobs: {exc}"})
    return pd.DataFrame(rows)

for schema_name, schema in schema_map.items():
    print(f"{schema_name}")
    display(recent_jobs(schema, status="error", limit=MAX_ROWS))


imaging


,timestamp,status,host,key_hash,error_message,key_preview,stack_preview
0,2026-01-22 15:23:27,error,tatchu3,6d6d9337d109362903c260162a2f2103,"DuplicateError: (""Duplicate entry 'sess9FY3GJQ...","{'session_id': 'sess9FY3GJQ3', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
1,2026-01-20 11:01:58,error,tatchu3,7a689f2f3d02bb1f16c890f79617755c,"DuplicateError: (""Duplicate entry 'sess9FXY0HR...","{'session_id': 'sess9FXY0HRX', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
2,2026-01-20 11:01:56,error,tatchu3,a2a77221c1127b777d62ff373714e3e1,"DuplicateError: (""Duplicate entry 'sess9FXY0HR...","{'session_id': 'sess9FXY0HRX', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
3,2026-01-20 10:52:26,error,tatchu3,070df51727a9d3916d266c772a54a94d,"DuplicateError: (""Duplicate entry 'sess9FXY0HR...","{'session_id': 'sess9FXY0HRX', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
4,2026-01-20 10:52:24,error,tatchu3,06165011d4c0e6e25fd56902da29b551,"DuplicateError: (""Duplicate entry 'sess9FXY0HR...","{'session_id': 'sess9FXY0HRX', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
5,2026-01-19 14:52:58,error,tatchu3,ceec8adf1e5d8c620290367bf56aa679,ValueError: no ROIs were found -- check regist...,"{'session_id': 'sess9FY9C1LF', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
6,2026-01-19 12:55:02,error,tatchu3,ada7831b91992d0b8fc393d6076d4d05,ValueError: the total number of frames should ...,"{'session_id': 'sess9FU7VMGS', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
7,2026-01-19 12:54:35,error,tatchu3,0cc58b8553d7d855ec8d772b90c383fb,ValueError: the total number of frames should ...,"{'session_id': 'sess9FU7WHSS', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
8,2026-01-19 12:54:08,error,tatchu3,78b02987b6ff7adeddf6fb3d4b863fab,ValueError: the total number of frames should ...,"{'session_id': 'sess9FU7XGAN', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
9,2026-01-19 12:53:41,error,tatchu3,12020d3fc5e33c092cc7326d3539ab8c,ValueError: the total number of frames should ...,"{'session_id': 'sess9FU7Y7DK', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."


model


,timestamp,status,host,key_hash,error_message,key_preview,stack_preview
0,2026-02-04 13:49:06,error,ibehavegpu1,b89638985ee53a269a17d83ad836668f,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
1,2026-02-04 13:49:06,error,ibehavegpu1,ca8d88f839649a2d281aa65ec5d3cb2a,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
2,2026-02-04 13:49:05,error,ibehavegpu1,535a4e8ee18ce41237f6d86d2e193404,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
3,2026-02-03 22:09:47,error,ibehavegpu1,cd29a6e24de7aa0a27169db3da5fd986,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9RZ0', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
4,2026-02-03 22:09:47,error,ibehavegpu1,eed275a29119e99b99d5251abc522a8d,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9RZ0', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
5,2026-02-03 22:09:46,error,ibehavegpu1,03000837905ffaa8c74a7b9163311cdf,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9RZ0', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
6,2026-02-03 21:10:40,error,ibehavegpu1,5ab9552e36f5329f9577193bb7756675,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9M00', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
7,2026-02-03 21:10:39,error,ibehavegpu1,0adcb3f62a9c1cb5ff81e52613bc2300,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9M00', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
8,2026-02-03 21:10:39,error,ibehavegpu1,12d99ea90969fe1823edabd8f0ea27bb,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYI9M00', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
9,2026-02-01 17:46:40,error,ibehavegpu1,ace543bb71760f9db8d09da775433910,ValueError: invalid literal for int() with bas...,"{'session_id': 'sess9FYHOFFQ', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."


denoising


""


mocap


,timestamp,status,host,key_hash,error_message,key_preview,stack_preview
0,2025-12-15 17:37:25,error,tatchu3,1b655a90b215db61535dded0c2d2c456,"DuplicateError: (""Duplicate entry 'motive_raw_...","{'session_id': 'sess9FU8IN4L', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
1,2025-12-15 16:49:35,error,tatchu3,7f3c740ce9c8dc657aec3cb4ada4344f,"DuplicateError: (""Duplicate entry 'motive_raw_...","{'session_id': 'sess9FU8JBWD', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
2,2025-12-15 15:06:24,error,tatchu3,3a3a2c6d1fd86038fd7bbb268afb230a,"OperationalError: (1205, 'Lock wait timeout ex...","{'session_id': 'sess9FU216UV', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
3,2025-12-15 14:52:45,error,tatchu3,c8729374b4d7e1a7e23f21cfef9e96b8,"DuplicateError: (""Duplicate entry 'motive_raw_...","{'session_id': 'sess9FU4D5W5', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
4,2025-12-15 14:50:16,error,tatchu3,4c5949c901c94214e4920322f5dd312b,"DuplicateError: (""Duplicate entry 'motive_raw_...","{'session_id': 'sess9FU4BT7I', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
5,2025-12-15 14:18:36,error,tatchu3,0fd35ba26a4f90d860a810b310dadf12,"DuplicateError: (""Duplicate entry 'motive_raw_...","{'session_id': 'sess9FU3PQIP', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
6,2025-12-10 09:30:29,error,tatchu3,0c3e11c40883cc06af547d81440939a7,"DuplicateError: (""Duplicate entry 'sess9FU1BEU...","{'session_id': 'sess9FU1BEUM', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
7,2025-12-09 21:14:55,error,tatchu3,e29b812e4f8311f9c305bfb38f984c7b,"DuplicateError: (""Duplicate entry 'Unlabeled 2...","{'session_id': 'sess9FUA9DEB', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
8,2025-12-09 21:14:19,error,tatchu3,b98fa4a1ed312d18e11eec02324bf8c8,"DuplicateError: (""Duplicate entry 'Unlabeled 2...","{'session_id': 'sess9FU9QCCF', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."
9,2025-12-09 21:02:01,error,tatchu3,a0b0111d3146c473022027469b6216bb,"DuplicateError: (""Duplicate entry 'Unlabeled 2...","{'session_id': 'sess9FU9OPGS', 'scan_id': 'sca...","Traceback (most recent call last):\n File ""/h..."



### 2b) Error keys by table (INITIALS + time range)

Fetch lowercase `key` from each jobs table and split results per populated table.


In [7]:

import ast

schema_module_map = {
    "imaging": imaging,
    "model": model,
    "denoising": denoising,
    "mocap": mocap,
}

def normalize_job_key(raw_key):
    if isinstance(raw_key, dict):
        return raw_key
    if isinstance(raw_key, str):
        try:
            parsed = ast.literal_eval(raw_key)
        except Exception:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}

def build_table_lookup(module):
    table_lookup = {}
    for attr in dir(module):
        obj = getattr(module, attr)
        if not hasattr(obj, "table_name"):
            continue
        if not hasattr(obj, "full_table_name"):
            continue
        table_name = getattr(obj, "table_name")
        table_lookup[table_name] = obj
        table_lookup[table_name.lstrip("_")] = obj
    return table_lookup

def build_dj_key(table_rel, key_dict):
    if table_rel is None:
        return {}, []
    pk_fields = list(table_rel.primary_key)
    dj_key = {field: key_dict.get(field) for field in pk_fields}
    missing = [field for field in pk_fields if field not in key_dict]
    return dj_key, missing

def relation_label(schema_name, table_name, table_rel):
    if table_rel is None:
        return f"{schema_name}.{table_name}"
    rel_name = getattr(table_rel, "__name__", None)
    if rel_name:
        return f"{schema_name}.{rel_name}"
    return f"{schema_name}.{table_name}"

session_scope = session.SessionUser * subject.User & f'initials = "{INITIALS}"'
session_scope &= session.Session & f'session_datetime >= "{DATE_FROM}"'
if DATE_TO:
    session_scope &= session.Session & f'session_datetime <= "{DATE_TO}"'

session_ids = set((session.Session & session_scope).fetch("session_id"))
print(f"Scoped sessions for {INITIALS}: {len(session_ids)}")

error_keys_by_table = {}
dj_keys_by_table = {}

for schema_name, schema in schema_map.items():
    table_lookup = build_table_lookup(schema_module_map[schema_name])
    try:
        error_rows = (schema.jobs & 'status="error"').fetch(
            "table_name",
            "timestamp",
            "key_hash",
            "key",
            as_dict=True,
            order_by="table_name, timestamp desc",
        )
    except Exception as exc:
        print(f"\n### {schema_name}: unavailable ({exc})")
        continue

    grouped_rows = {}
    for row in error_rows:
        key_dict = normalize_job_key(row.get("key"))
        if key_dict.get("session_id") not in session_ids:
            continue

        table_name = row.get("table_name", "<unknown_table>")
        table_rel = table_lookup.get(table_name) or table_lookup.get(table_name.lstrip("_"))
        dj_key, missing_pk = build_dj_key(table_rel, key_dict)
        table_label = relation_label(schema_name, table_name, table_rel)

        grouped_rows.setdefault(table_name, []).append(
            {
                "table_name": table_name,
                "timestamp": row.get("timestamp"),
                "key_hash": row.get("key_hash"),
                "key": key_dict,
                "dj_key": dj_key,
                "missing_pk_fields": missing_pk,
            }
        )

        if dj_key and not missing_pk:
            dj_keys_by_table.setdefault(table_label, []).append(dj_key)

    if not grouped_rows:
        print(f"\n### {schema_name}: no matching error keys")
        continue

    print(f"\n### {schema_name}")
    for table_name in sorted(grouped_rows):
        table_rel = table_lookup.get(table_name) or table_lookup.get(table_name.lstrip("_"))
        table_label = relation_label(schema_name, table_name, table_rel)
        table_df = pd.DataFrame(grouped_rows[table_name]).reset_index(drop=True)
        error_keys_by_table[table_label] = table_df
        print(f"{table_label}: {len(table_df)}")
        display(table_df.head(MAX_ROWS))

for table_label, keys in list(dj_keys_by_table.items()):
    dj_keys_by_table[table_label] = pd.DataFrame(keys).drop_duplicates().to_dict("records")

print(f"\nerror_keys_by_table tables: {len(error_keys_by_table)}")
print(f"dj_keys_by_table tables: {len(dj_keys_by_table)}")


Scoped sessions for SM: 362

### imaging
imaging.Processing: 65


,table_name,timestamp,key_hash,key,dj_key,missing_pk_fields
0,__processing,2026-01-14 15:31:31,53b7e777cef0831d2867bc11a4af6cb8,"{'session_id': 'sess9FX18K7G', 'scan_id': 'sca...","{'session_id': 'sess9FX18K7G', 'scan_id': 'sca...",[]
1,__processing,2026-01-14 15:29:56,da089cebe04cd837970a311c2adaf9e6,"{'session_id': 'sess9FX18EMU', 'scan_id': 'sca...","{'session_id': 'sess9FX18EMU', 'scan_id': 'sca...",[]
2,__processing,2026-01-14 15:28:27,419267011fcb86fe86aff049f2515b2c,"{'session_id': 'sess9FX18XKS', 'scan_id': 'sca...","{'session_id': 'sess9FX18XKS', 'scan_id': 'sca...",[]
3,__processing,2026-01-14 15:26:55,9c7f9c0cebbab5fed73458b5d0f9cdb8,"{'session_id': 'sess9FX18PQT', 'scan_id': 'sca...","{'session_id': 'sess9FX18PQT', 'scan_id': 'sca...",[]
4,__processing,2026-01-14 15:14:19,3586e4a157f20742d2bd29836e02e545,"{'session_id': 'sess9FWTI411', 'scan_id': 'sca...","{'session_id': 'sess9FWTI411', 'scan_id': 'sca...",[]
5,__processing,2026-01-14 15:11:32,33cfa2c5b157872e87b3699f06d81355,"{'session_id': 'sess9FWTHUDQ', 'scan_id': 'sca...","{'session_id': 'sess9FWTHUDQ', 'scan_id': 'sca...",[]
6,__processing,2026-01-14 15:08:50,dd781d3f00fc73e42aacec9d219aacd7,"{'session_id': 'sess9FWTHLWS', 'scan_id': 'sca...","{'session_id': 'sess9FWTHLWS', 'scan_id': 'sca...",[]
7,__processing,2026-01-14 15:05:58,7577aa9c9958159d4e92913c0d274d13,"{'session_id': 'sess9FWTHB7O', 'scan_id': 'sca...","{'session_id': 'sess9FWTHB7O', 'scan_id': 'sca...",[]
8,__processing,2026-01-14 15:02:21,5cb2a896060dd36d7f25d9f8a3589d9c,"{'session_id': 'sess9FWPYEXJ', 'scan_id': 'sca...","{'session_id': 'sess9FWPYEXJ', 'scan_id': 'sca...",[]
9,__processing,2026-01-14 15:00:53,5481adb88f421664fb571ba0633db6c8,"{'session_id': 'sess9FWPY5L8', 'scan_id': 'sca...","{'session_id': 'sess9FWPY5L8', 'scan_id': 'sca...",[]



### model
model.PoseEstimationNew: 3


,table_name,timestamp,key_hash,key,dj_key,missing_pk_fields
0,__pose_estimation_new,2026-02-04 13:49:06,b89638985ee53a269a17d83ad836668f,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...",[]
1,__pose_estimation_new,2026-02-04 13:49:06,ca8d88f839649a2d281aa65ec5d3cb2a,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...",[]
2,__pose_estimation_new,2026-02-04 13:49:05,535a4e8ee18ce41237f6d86d2e193404,"{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...","{'session_id': 'sess9FYIUOF1', 'scan_id': 'sca...",[]



### denoising: no matching error keys

### mocap: no matching error keys

error_keys_by_table tables: 2
dj_keys_by_table tables: 2


[{'session_id': 'sess9FX18K7G', 'scan_id': 'scan9FX18K7G', 'paramset_idx': 0},
 {'session_id': 'sess9FX18EMU', 'scan_id': 'scan9FX18EMU', 'paramset_idx': 0},
 {'session_id': 'sess9FX18XKS', 'scan_id': 'scan9FX18XKS', 'paramset_idx': 0},
 {'session_id': 'sess9FX18PQT', 'scan_id': 'scan9FX18PQT', 'paramset_idx': 0},
 {'session_id': 'sess9FWTI411', 'scan_id': 'scan9FWTI411', 'paramset_idx': 0},
 {'session_id': 'sess9FWTHUDQ', 'scan_id': 'scan9FWTHUDQ', 'paramset_idx': 0},
 {'session_id': 'sess9FWTHLWS', 'scan_id': 'scan9FWTHLWS', 'paramset_idx': 0},
 {'session_id': 'sess9FWTHB7O', 'scan_id': 'scan9FWTHB7O', 'paramset_idx': 0},
 {'session_id': 'sess9FWPYEXJ', 'scan_id': 'scan9FWPYEXJ', 'paramset_idx': 0},
 {'session_id': 'sess9FWPY5L8', 'scan_id': 'scan9FWPY5L8', 'paramset_idx': 0},
 {'session_id': 'sess9FWPY01P', 'scan_id': 'scan9FWPY01P', 'paramset_idx': 0},
 {'session_id': 'sess9FWPXTZA', 'scan_id': 'scan9FWPXTZA', 'paramset_idx': 0},
 {'session_id': 'sess9FX4RRN9', 'scan_id': 'scan9FX4


## 3) Stranded task snapshots

`task AND NOT populated output` for all task-backed population tables.


In [8]:

modules_for_task_checks = {
    "imaging": imaging,
    "model": model,
    "denoising": denoising,
    "mocap": mocap,
    "disk": disk,
}

task_output_pairs = []
for module_name, module in modules_for_task_checks.items():
    if module is None:
        continue

    task_attrs = sorted(name for name in dir(module) if "Task" in name)
    for task_attr in task_attrs:
        task_table = getattr(module, task_attr)
        if not (hasattr(task_table, "full_table_name") and hasattr(task_table, "__and__")):
            continue

        output_attr = task_attr.replace("Task", "")
        if output_attr == task_attr or not hasattr(module, output_attr):
            continue

        output_table = getattr(module, output_attr)
        if not (hasattr(output_table, "full_table_name") and hasattr(output_table, "__and__")):
            continue

        task_output_pairs.append(
            {
                "module": module_name,
                "task_name": task_attr,
                "output_name": output_attr,
                "task_table": task_table,
                "output_table": output_table,
            }
        )

if not task_output_pairs:
    print("No task/output pairs found.")
else:
    for pair in task_output_pairs:
        label = f"{pair['module']}.{pair['task_name']} -> {pair['module']}.{pair['output_name']}"
        try:
            stranded_query = pair["task_table"] & dj.Not(pair["output_table"])
            print(f"{label}: {len(stranded_query)}")
        except Exception as exc:
            print(f"{label}: unavailable ({exc})")


imaging.ProcessingTask -> imaging.Processing: 104
model.DLCLivePoseEstimationTask -> model.DLCLivePoseEstimation: 280
model.PoseEstimationTask -> model.PoseEstimation: 0
model.PoseEstimationTaskNew -> model.PoseEstimationNew: 6
denoising.DenoisingTask -> denoising.Denoising: 0
mocap.MotionCaptureTask -> mocap.MotionCapture: 3
disk.DLCImputationTask -> disk.DLCImputation: 0
disk.MocapImputationTask -> disk.MocapImputation: 0



## 4) Candidate key selection (user/date filter)


In [9]:

user_key = (session.SessionUser * subject.User & f'initials = "{INITIALS}"').fetch("KEY")
time_query = session.Session & f'session_datetime >= "{DATE_FROM}"'
if DATE_TO:
    time_query &= f'session_datetime <= "{DATE_TO}"'
time_key = time_query.fetch("KEY")

candidate_keys = (session.Session & user_key & time_key).fetch("KEY")
print("candidate sessions:", len(candidate_keys))

candidate_df = pd.DataFrame(candidate_keys)
candidate_df.head(20)


candidate sessions: 362


,session_id
0,sess9FT2Z540
1,sess9FT3EPO7
2,sess9FT462SV
3,sess9FT46BBS
4,sess9FT6L42R
5,sess9FT75UGB
6,sess9FT7PJFI
7,sess9FT8C6I9
8,sess9FT8T8F3
9,sess9FTALH2G


In [10]:

output_csv = repo_root / "notebooks" / "tmp_cleanup_candidate_keys.csv"
pd.DataFrame(candidate_keys).to_csv(output_csv, index=False)
print(f"Wrote candidate key preview to {output_csv}")


Wrote candidate key preview to /Users/tobiasr/Documents/GitHub/troselab/adamacs_ingest/notebooks/tmp_cleanup_candidate_keys.csv



## 5) Optional write block (disabled)

Keep this disabled for normal use. The examples below are intentionally commented.


In [11]:

def require_write_mode():
    if not ALLOW_DB_WRITES:
        raise RuntimeError(
            "Database write mode is disabled. Set ALLOW_DB_WRITES=True only in controlled maintenance windows."
        )

# Example cleanup snippets (DO NOT RUN unless explicitly approved):
# require_write_mode()
# cleanup_key = {"session_id": "sessXXXX", "scan_id": "scanXXXX"}
# (imaging.schema.jobs & 'status="error"' & cleanup_key).delete()
# (model.schema.jobs & 'status="error"' & cleanup_key).delete()
# (imaging.ProcessingTask & cleanup_key & dj.Not(imaging.Processing)).delete()
